In [1]:
from dotenv import load_dotenv
import os

In [2]:
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
OPENAI_BASE_URL = os.getenv("OPENAI_API_BASE")
OPENAI_MODEL = os.getenv("OPENAI_MODEL")
OPENAI_BASE_URL, OPENAI_MODEL, OPENAI_API_KEY, 

(None, 'gpt-5.4-nano', 'sk-3CDnIjNIBoqFT7DNvzuGF9NXPqNkwQO8')

In [33]:
# Cell 1 — imports

from __future__ import annotations

import json
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from typing_extensions import TypedDict
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI

from langfuse import get_client
from langfuse.langchain import CallbackHandler

from rdkit import Chem
from rdkit.Chem import inchi
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import Descriptors
from rdkit.Chem import Crippen
from rdkit.Chem import Lipinski


from functools import lru_cache

In [34]:
from langfuse import Langfuse

langfuse = Langfuse()
print(langfuse.auth_check())  # should return True

from langfuse.langchain import CallbackHandler
langfuse_handler = CallbackHandler()  # reads env vars

True


In [35]:
# Cell 2 — model + tracing setup
# load_env() is already done, per your note.

LF_CONFIG = {
    "callbacks": [langfuse_handler],
}

gen_llm = ChatOpenAI(
    model=OPENAI_MODEL,
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    temperature=0.2,
)

extract_base_llm = ChatOpenAI(
    model=OPENAI_MODEL,
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    temperature=0.0,
)

langfuse = get_client()
langfuse_handler = CallbackHandler()

In [36]:
# Cell 3 — markdown output configuration

# We no longer use structured output.
# The generation model is instructed to return a protocol directly in Markdown
# with fixed sections that are easier to render and inspect downstream.



In [37]:
# Cell 4 — helpers

ENABLE_PUBCHEM_IUPAC = False  # set True only if you want online PubChem enrichment


def message_to_text(msg: Any) -> str:
    content = getattr(msg, "content", msg)
    if isinstance(content, str):
        return content
    return json.dumps(content, ensure_ascii=False)


def ensure_parent(path: str | Path) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    return path


def split_reaction_smiles(reaction_smiles: str) -> dict[str, list[str] | str]:
    parts = reaction_smiles.strip().split(">")
    if len(parts) != 3:
        raise ValueError(
            "Expected reaction SMILES with 3 parts: reactants>agents>products. "
            f"Got {len(parts)} parts."
        )

    reactants_part, agents_part, products_part = parts

    def split_side(side: str) -> list[str]:
        side = side.strip()
        if not side:
            return []
        return [frag.strip() for frag in side.split(".") if frag.strip()]

    return {
        "reactants_part": reactants_part,
        "agents_part": agents_part,
        "products_part": products_part,
        "reactants": split_side(reactants_part),
        "agents": split_side(agents_part),
        "products": split_side(products_part),
    }


def clear_atom_maps(mol: Chem.Mol) -> Chem.Mol:
    mol = Chem.Mol(mol)
    for atom in mol.GetAtoms():
        if atom.HasProp("molAtomMapNumber"):
            atom.ClearProp("molAtomMapNumber")
    return mol


def standardize_mol(mol: Chem.Mol) -> Chem.Mol:
    mol = Chem.Mol(mol)
    mol = rdMolStandardize.Cleanup(mol)
    mol = rdMolStandardize.FragmentParent(mol)

    uncharger = rdMolStandardize.Uncharger()
    mol = uncharger.uncharge(mol)

    tautomer_enumerator = rdMolStandardize.TautomerEnumerator()
    mol = tautomer_enumerator.Canonicalize(mol)

    Chem.SanitizeMol(mol)
    return mol


@lru_cache(maxsize=10000)
def maybe_pubchem_iupac(smiles: str) -> str | None:
    if not ENABLE_PUBCHEM_IUPAC or pcp is None:
        return None

    try:
        rows = pcp.get_properties(
            ["IUPACName"],
            smiles,
            "smiles",
        )
        if rows and isinstance(rows, list):
            value = rows[0].get("IUPACName")
            return value or None
    except Exception:
        return None

    return None


def serialize_molecule(smiles: str, side: str, idx: int) -> dict:
    mol_in = Chem.MolFromSmiles(smiles)
    if mol_in is None:
        return {
            "side": side,
            "index": idx,
            "input_smiles": smiles,
            "valid": False,
            "error": "invalid_smiles",
        }

    try:
        mapped_canonical_smiles = Chem.MolToSmiles(
            mol_in,
            canonical=True,
            isomericSmiles=True,
        )

        mol_unmapped = clear_atom_maps(mol_in)
        mol_std = standardize_mol(mol_unmapped)

        canonical_smiles = Chem.MolToSmiles(
            mol_std,
            canonical=True,
            isomericSmiles=True,
        )

        cxsmiles = Chem.MolToCXSmiles(mol_std)
        smarts = Chem.MolToSmarts(mol_std)
        inchi_str = inchi.MolToInchi(mol_std)
        inchikey = inchi.MolToInchiKey(mol_std)
        formula = rdMolDescriptors.CalcMolFormula(mol_std)
        exact_mw = rdMolDescriptors.CalcExactMolWt(mol_std)

        return {
            "side": side,
            "index": idx,
            "input_smiles": smiles,
            "valid": True,
            "mapped_canonical_smiles": mapped_canonical_smiles,
            "canonical_smiles": canonical_smiles,
            "cxsmiles": cxsmiles,
            "smarts": smarts,
            "inchi": inchi_str,
            "inchikey": inchikey,
            "formula": formula,
            "iupac_name": maybe_pubchem_iupac(canonical_smiles),
            "molblock": Chem.MolToMolBlock(mol_std),
            "descriptors": {
                "mol_wt": Descriptors.MolWt(mol_std),
                "exact_mw": exact_mw,
                "logp": Crippen.MolLogP(mol_std),
                "tpsa": rdMolDescriptors.CalcTPSA(mol_std),
                "hba": Lipinski.NumHAcceptors(mol_std),
                "hbd": Lipinski.NumHDonors(mol_std),
                "rotatable_bonds": Lipinski.NumRotatableBonds(mol_std),
                "ring_count": rdMolDescriptors.CalcNumRings(mol_std),
                "heavy_atom_count": mol_std.GetNumHeavyAtoms(),
            },
        }

    except Exception as e:
        return {
            "side": side,
            "index": idx,
            "input_smiles": smiles,
            "valid": False,
            "error": f"{type(e).__name__}: {e}",
        }


def reaction_notations(reaction_smiles: str) -> dict:
    out = {
        "input_reaction_smiles": reaction_smiles,
        "canonical_mapped_reaction_smiles": None,
        "canonical_unmapped_reaction_smiles": None,
        "cx_reaction_smiles": None,
        "rxn_block_v2000": None,
    }

    try:
        rxn = rdChemReactions.ReactionFromSmiles(reaction_smiles)
        out["canonical_mapped_reaction_smiles"] = rdChemReactions.ReactionToSmiles(rxn, True)
        out["cx_reaction_smiles"] = rdChemReactions.ReactionToCXSmiles(rxn, True)
        out["rxn_block_v2000"] = rdChemReactions.ReactionToRxnBlock(rxn)

        rxn_unmapped = rdChemReactions.ReactionFromSmiles(reaction_smiles)
        rdChemReactions.RemoveMappingNumbersFromReactions(rxn_unmapped)
        out["canonical_unmapped_reaction_smiles"] = rdChemReactions.ReactionToSmiles(rxn_unmapped, True)

    except Exception:
        pass

    return out


def brief_component_lines(items: list[dict]) -> str:
    lines = []
    for item in items:
        if not item.get("valid"):
            lines.append(f"- invalid_smiles={item['input_smiles']}")
            continue

        label = item.get("iupac_name") or item["canonical_smiles"]
        lines.append(
            f"- name={label}; "
            f"canonical_smiles={item['canonical_smiles']}; "
            f"formula={item['formula']}; "
            f"inchikey={item['inchikey']}"
        )
    return "\n".join(lines)


def build_protocol_prompt(state: dict) -> str:
    reactant_lines = brief_component_lines(state["reactant_records"])
    agent_lines = brief_component_lines(state["agent_records"])
    product_lines = brief_component_lines(state["product_records"])

    return f"""
You are a senior synthetic chemist.

Your task is to infer a plausible laboratory synthesis protocol from the reaction transformation.

Important formatting rules:
- Return Markdown only.
- Do not return JSON.
- Do not mention atom mapping numbers.
- Use unmapped SMILES only if you need to reference structures.
- Follow the exact section structure below.

Required Markdown structure:

# Protocol Name
A concise protocol title.

## Precursors
- **compound / material** — **amount, equivalents, or loading if inferable; otherwise write "amount not specified"** — role or short note

## Step-by-step actions
1. Ordered experimental actions for carrying out the transformation.
2. Include reagent charging, mixing, temperature control, reaction progress monitoring, quench or workup only when chemically appropriate.
3. Keep the steps practical and chemically coherent.

## Conditions
- **Temperature:** ...
- **Time:** ...
- **Pressure:** ...
- **Atmosphere:** ...
- **Solvent:** ...
- **Catalyst / additive:** ...
- **Monitoring / endpoint:** ...

## Comments
- Explain the chemistry-based rationale for the chosen protocol.
- Mention selectivity considerations, likely roles of reagents, and key mechanistic logic.
- Mention ambiguities or plausible alternatives when the SMIRKS alone is insufficient.

Reaction summary:
- Unmapped reaction SMIRKS: {state["reaction_meta"]["canonical_unmapped_reaction_smiles"]}

Reactants:
{reactant_lines}

Agents:
{agent_lines if agent_lines else "- none"}

Products:
{product_lines}

Guidance:
- Infer a realistic protocol style from the transformation type.
- Prefer chemically typical solvents, bases, catalysts, and temperatures for this transformation.
- If exact quantities cannot be known from SMIRKS, state that clearly and use equivalents / catalyst loadings / qualitative amounts when reasonable.
- Be explicit about uncertainty rather than inventing unsupported specifics.
""".strip()


def load_existing_ids(path: str | Path) -> set[str]:
    path = Path(path)
    if not path.exists():
        return set()

    ids = set()
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                record_id = obj.get("id")
                if record_id:
                    ids.add(record_id)
            except Exception:
                pass
    return ids


def append_jsonl(record: dict, path: str | Path) -> None:
    path = ensure_parent(path)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")



In [38]:
# Cell 5 — graph state

class ReactionState(TypedDict, total=False):
    reaction_smiles: str
    output_path: str

    split_parts: dict
    reaction_meta: dict

    reactant_records: list[dict]
    agent_records: list[dict]
    product_records: list[dict]
    invalid_records: list[dict]

    protocol_prompt: str
    protocol_markdown: str

    jsonl_record: dict



In [39]:
# Cell 6 — graph nodes

def node_split_reaction(state: ReactionState) -> dict:
    parts = split_reaction_smiles(state["reaction_smiles"])
    meta = reaction_notations(state["reaction_smiles"])
    return {
        "split_parts": parts,
        "reaction_meta": meta,
    }


def node_normalize_components(state: ReactionState) -> dict:
    split_parts = state["split_parts"]

    reactant_records = [
        serialize_molecule(smi, "reactant", i)
        for i, smi in enumerate(split_parts["reactants"])
    ]
    agent_records = [
        serialize_molecule(smi, "agent", i)
        for i, smi in enumerate(split_parts["agents"])
    ]
    product_records = [
        serialize_molecule(smi, "product", i)
        for i, smi in enumerate(split_parts["products"])
    ]

    invalid_records = [
        rec
        for rec in (reactant_records + agent_records + product_records)
        if not rec.get("valid", False)
    ]

    return {
        "reactant_records": reactant_records,
        "agent_records": agent_records,
        "product_records": product_records,
        "invalid_records": invalid_records,
    }


def node_build_prompt(state: ReactionState) -> dict:
    prompt = build_protocol_prompt(state)
    return {"protocol_prompt": prompt}


def node_run_llm_analysis(state: ReactionState) -> dict:
    resp = gen_llm.invoke(state["protocol_prompt"], config=LF_CONFIG)
    return {"protocol_markdown": message_to_text(resp)}


def node_save_jsonl(state: ReactionState) -> dict:
    record = {
        "id": str(uuid.uuid4()),
        "created_at": datetime.now(timezone.utc).isoformat(),
        "task_type": "reaction_protocol_generation_from_smirks",
        "model": OPENAI_MODEL,
        "reaction": {
            "input_reaction_smiles": state["reaction_smiles"],
            "reaction_meta": state["reaction_meta"],
            "split_parts": {
                "reactants": state["split_parts"]["reactants"],
                "agents": state["split_parts"]["agents"],
                "products": state["split_parts"]["products"],
            },
        },
        "components": {
            "reactants": state["reactant_records"],
            "agents": state["agent_records"],
            "products": state["product_records"],
            "invalid": state["invalid_records"],
        },
        "protocol": {
            "prompt": state["protocol_prompt"],
            "markdown": state["protocol_markdown"],
        },
    }

    append_jsonl(record, state["output_path"])
    return {"jsonl_record": record}



In [40]:
# Cell 7 — build graph

builder = StateGraph(ReactionState)

builder.add_node("split_reaction", node_split_reaction)
builder.add_node("normalize_components", node_normalize_components)
builder.add_node("build_prompt", node_build_prompt)
builder.add_node("run_llm_analysis", node_run_llm_analysis)
builder.add_node("save_jsonl", node_save_jsonl)

builder.add_edge(START, "split_reaction")
builder.add_edge("split_reaction", "normalize_components")
builder.add_edge("normalize_components", "build_prompt")
builder.add_edge("build_prompt", "run_llm_analysis")
builder.add_edge("run_llm_analysis", "save_jsonl")
builder.add_edge("save_jsonl", END)

graph = builder.compile()



In [41]:
# Cell 8 — convenience wrapper

def run_reaction(
    reaction_smiles: str,
    *,
    output_path: str = "data/reactions_protocol_markdown.jsonl",
) -> dict:
    initial_state: ReactionState = {
        "reaction_smiles": reaction_smiles,
        "output_path": output_path,
    }

    result = graph.invoke(initial_state)
    langfuse.flush()
    return result



In [42]:
# Cell 9 — example run

reaction = (
    "C1CCOC1.CCCCCC.Cl[C:11](=[O:3])[CH:5]=[C:12]([CH3:1])[CH3:2]."
    "[Br:4][c:13]1[cH:6][cH:8][c:14]([NH2:10])[cH:9][cH:7]1"
    ">>"
    "[CH3:1][C:12]([CH3:2])=[CH:5][C:11](=[O:3])[NH:10][c:14]1[cH:8][cH:6]"
    "[c:13]([Br:4])[cH:7][cH:9]"
)

result = run_reaction(reaction)

print("Reactants:", len(result["reactant_records"]))
print("Agents:", len(result["agent_records"]))
print("Products:", len(result["product_records"]))
print("Invalid molecules:", len(result["invalid_records"]))



[22:06:11] Initializing MetalDisconnector
[22:06:11] Running MetalDisconnector
[22:06:11] Initializing Normalizer
[22:06:11] Running Normalizer
[22:06:11] Initializing MetalDisconnector
[22:06:11] Running MetalDisconnector
[22:06:11] Initializing Normalizer
[22:06:11] Running Normalizer
[22:06:11] Running LargestFragmentChooser
[22:06:11] Running Uncharger
[22:06:11] Initializing MetalDisconnector
[22:06:11] Running MetalDisconnector
[22:06:11] Initializing Normalizer
[22:06:11] Running Normalizer
[22:06:11] Initializing MetalDisconnector
[22:06:11] Running MetalDisconnector
[22:06:11] Initializing Normalizer
[22:06:11] Running Normalizer
[22:06:11] Running LargestFragmentChooser
[22:06:11] Running Uncharger
[22:06:11] Initializing MetalDisconnector
[22:06:11] Running MetalDisconnector
[22:06:11] Initializing Normalizer
[22:06:11] Running Normalizer
[22:06:11] Initializing MetalDisconnector
[22:06:11] Running MetalDisconnector
[22:06:11] Initializing Normalizer
[22:06:11] Running Norma

Reactants: 4
Agents: 0
Products: 1
Invalid molecules: 1


In [43]:
result


{'reaction_smiles': 'C1CCOC1.CCCCCC.Cl[C:11](=[O:3])[CH:5]=[C:12]([CH3:1])[CH3:2].[Br:4][c:13]1[cH:6][cH:8][c:14]([NH2:10])[cH:9][cH:7]1>>[CH3:1][C:12]([CH3:2])=[CH:5][C:11](=[O:3])[NH:10][c:14]1[cH:8][cH:6][c:13]([Br:4])[cH:7][cH:9]',
 'output_path': 'data/reactions_protocol_markdown.jsonl',
 'split_parts': {'reactants_part': 'C1CCOC1.CCCCCC.Cl[C:11](=[O:3])[CH:5]=[C:12]([CH3:1])[CH3:2].[Br:4][c:13]1[cH:6][cH:8][c:14]([NH2:10])[cH:9][cH:7]1',
  'agents_part': '',
  'products_part': '[CH3:1][C:12]([CH3:2])=[CH:5][C:11](=[O:3])[NH:10][c:14]1[cH:8][cH:6][c:13]([Br:4])[cH:7][cH:9]',
  'reactants': ['C1CCOC1',
   'CCCCCC',
   'Cl[C:11](=[O:3])[CH:5]=[C:12]([CH3:1])[CH3:2]',
   '[Br:4][c:13]1[cH:6][cH:8][c:14]([NH2:10])[cH:9][cH:7]1'],
  'agents': [],
  'products': ['[CH3:1][C:12]([CH3:2])=[CH:5][C:11](=[O:3])[NH:10][c:14]1[cH:8][cH:6][c:13]([Br:4])[cH:7][cH:9]']},
 'reaction_meta': {'input_reaction_smiles': 'C1CCOC1.CCCCCC.Cl[C:11](=[O:3])[CH:5]=[C:12]([CH3:1])[CH3:2].[Br:4][c:13]1[cH:6][c

In [44]:
result.keys()

dict_keys(['reaction_smiles', 'output_path', 'split_parts', 'reaction_meta', 'reactant_records', 'agent_records', 'product_records', 'invalid_records', 'protocol_prompt', 'protocol_markdown', 'jsonl_record'])

In [45]:
result['protocol_markdown']

'# Protocol Name\nSynthesis of an Amide via Acyl Chloride Coupling with an Anilide (Using 1,1-Ring Ether and Alkyl Chain as Reactants)\n\n## Precursors\n- **1,3-dioxolane (C1CCOC1)** — amount not specified — oxygenated cyclic ether (solvent or co-substrate depending on intended transformation)\n- **hexane (CCCCCC)** — amount not specified — hydrophobic solvent/co-solvent or reagent (unclear role from provided information)\n- **2-methylacryloyl chloride (CC(C)=CC(=O)Cl)** — amount not specified — acyl chloride electrophile\n- **bromoaniline (Nc1ccc(Br)cc1)** — amount not specified — aniline nucleophile to form an amide\n- **base (e.g., triethylamine or pyridine)** — amount not specified — acid scavenger (required to neutralize HCl; not listed but typically necessary)\n- **inert drying agent (optional, e.g., molecular sieves)** — amount not specified — to suppress hydrolysis of acyl chloride (optional)\n\n## Step-by-step actions\n1. **Set up an anhydrous reaction**: Assemble a dry flask 

In [46]:
from IPython.display import Markdown, display
display(Markdown(result['protocol_markdown']))


# Protocol Name
Synthesis of an Amide via Acyl Chloride Coupling with an Anilide (Using 1,1-Ring Ether and Alkyl Chain as Reactants)

## Precursors
- **1,3-dioxolane (C1CCOC1)** — amount not specified — oxygenated cyclic ether (solvent or co-substrate depending on intended transformation)
- **hexane (CCCCCC)** — amount not specified — hydrophobic solvent/co-solvent or reagent (unclear role from provided information)
- **2-methylacryloyl chloride (CC(C)=CC(=O)Cl)** — amount not specified — acyl chloride electrophile
- **bromoaniline (Nc1ccc(Br)cc1)** — amount not specified — aniline nucleophile to form an amide
- **base (e.g., triethylamine or pyridine)** — amount not specified — acid scavenger (required to neutralize HCl; not listed but typically necessary)
- **inert drying agent (optional, e.g., molecular sieves)** — amount not specified — to suppress hydrolysis of acyl chloride (optional)

## Step-by-step actions
1. **Set up an anhydrous reaction**: Assemble a dry flask with a stir bar, septum, and inert gas (N₂ or Ar). Dry glassware is recommended because **acyl chlorides hydrolyze readily**.
2. **Charge solvent/media**: Add **anhydrous 1,3-dioxolane** (and/or the indicated **hexane** as co-solvent if needed for solubility). Cool to **0–5 °C**.
3. **Dissolve the nucleophile**: Add **bromoaniline** to the cooled solvent and stir until homogeneous.
4. **Add base**: Add a suitable **base** (typically triethylamine or pyridine) at **~1–2 equivalents relative to the acyl chloride** to capture the HCl generated during amide formation.
5. **Acyl chloride addition**: Prepare a solution of **2-methylacryloyl chloride** in the same solvent (or add neat if safe and compatible). **Add dropwise** to the stirred aniline/base solution while maintaining **0–5 °C**.
6. **Allow reaction to warm**: After complete addition, gradually warm to **room temperature** and continue stirring.
7. **Monitor progress**: Track consumption of the acyl chloride/aniline by **TLC** (if applicable) and/or **LC-MS** / **IR** (disappearance of acid chloride carbonyl stretch around ~1800 cm⁻¹ in IR).
8. **Quench/workup**: Once complete, carefully **quench residual reactive acyl chloride** by stirring briefly with **ice-cold water** (or dilute aqueous base) while maintaining good mixing. Then separate layers if biphasic.
9. **Extract/purify**: Extract the organic phase, wash with water and possibly brine, then dry (Na₂SO₄ or MgSO₄). Concentrate and purify (commonly **column chromatography** or recrystallization, depending on the product polarity).
10. **Confirm product**: Verify structure and purity by **NMR** and **MS**. The product indicated contains an **amide linkage** and a **bromo-substituted anilide aromatic ring**.

## Conditions
- **Temperature:** 0–5 °C during acyl chloride addition; then room temperature
- **Time:** amount not specified (typical: 1–4 h after addition; longer if low solubility)
- **Pressure:** atmospheric
- **Atmosphere:** inert (N₂ or Ar), dry conditions
- **Solvent:** 1,3-dioxolane (and hexane as co-solvent if solubility/phase behavior requires)
- **Catalyst / additive:** none specified; **base/additive not listed but strongly recommended** (triethylamine/pyridine or similar)
- **Monitoring / endpoint:** disappearance of acyl chloride (IR/TLC/LC-MS), or cessation of conversion on LC-MS

## Comments
- The transformation implied by the inputs is a **classic amide coupling**: an **aniline nucleophile** reacts with an **acyl chloride** to give an **amide**, with **HCl** as the byproduct.
- **Acyl chloride handling:** Because 2-methylacryloyl chloride is moisture-sensitive, the protocol is designed to **minimize water exposure** (inert atmosphere, dry solvent, chilled addition).
- **Role of the base:** Although not listed, a base is practically necessary to avoid protonating the aniline and to **neutralize the released HCl**, preventing reduced coupling efficiency and possible side reactions (e.g., decomposition/hydrolysis of the acyl chloride).
- **Selectivity considerations:** Direct acylation of anilines typically gives the desired **N-acyl aniline**. However, if the environment allows competing reactions (e.g., hydrolysis, polymerization tendencies from α,β-unsaturated acyl systems), low temperature addition and controlled mixing help.
- **Uncertainty about 1,3-dioxolane and hexane roles:** The provided reactant list includes both **1,3-dioxolane** and **hexane**, but the reaction summary does not uniquely define whether they are **solvents** or **co-reactants**. The most chemically consistent interpretation is that they serve as **solvent/co-solvent media** to dissolve the aniline and manage viscosity/polarity. If your actual target does include incorporation of these components, the pathway would differ; with the product description resembling an **amide bearing a substituted aromatic ring**, the primary coupling step is still the **aniline + acyl chloride → amide**.
- **Plausible alternative bases:** DIPEA, pyridine, or the aniline itself in excess can sometimes serve as acid scavenger, but using a dedicated base generally improves reproducibility.

If you tell me the intended full product name (or confirm whether 1,3-dioxolane/hexane are merely solvents), I can tighten the protocol with more specific solvent choice, equivalents, and purification strategy.

In [47]:
result['invalid_records']

[{'side': 'product',
  'index': 0,
  'input_smiles': '[CH3:1][C:12]([CH3:2])=[CH:5][C:11](=[O:3])[NH:10][c:14]1[cH:8][cH:6][c:13]([Br:4])[cH:7][cH:9]',
  'valid': False,
  'error': 'invalid_smiles'}]